# Stage 1: a pretrained recurrent backbone with an added attention route

**Run with Runtime → Run all. Nothing to set.** Reconnect and run all again after a disconnect;
finished pieces load from Drive.

## The plan, and what this stage decides

The Falcon screen showed that pretrained hybrids have *composed* routes: attention and the SSM are
components of one computation, not substitutes. So instead we start from a pretrained **pure
Mamba** language model — a recurrent route that is viable by construction — and add a parallel
attention route ourselves. That reproduces the paper's two-route setting at roughly 370M
parameters with a real pretrained recurrent model.

Stage 1 answers the two questions that decide whether the full experiment is worth building:

1. **Is the recurrent route viable, and where does it stop being viable?** The frozen backbone is
   screened on in-context recall with natural-text filler: one passkey at several distances, then
   2, 4 and 8 key–value pairs. This gives the positive and negative conditions *before* any
   intervention, exactly as the paper's boundary does.
2. **Does the added attention route take over?** A parallel attention branch is inserted at evenly
   spaced layers, its output added to the residual stream, and its output projection initialised to
   zero so the model starts numerically unchanged. Only the new attention parameters train. If
   attention takes over a task the backbone can already do alone, that is the paper's
   "free retrieval is the default" result in a pretrained 370M model, and it is the starting point
   the four-arm experiment needs.

**Stage 2 (not in this notebook)** runs never / late / maintained / withdrawn from the model this
stage produces, pricing attention and asking whether computation returns to the backbone at the
viable loads but not beyond them.

## Structure

Every block is wrapped so its two contributions are separable:

```
h ← h + ssm_delta(h) + attn(h)
```

`ssm_delta` is the pretrained block's own residual contribution; `attn` is the new route (zero at
every non-selected layer). Both can be recorded, zeroed or resampled from another sequence, which
is how causal dependence is measured, as in the paper.

## Speed

The notebook first tries to install the fast Mamba kernels (`causal-conv1d`, `mamba-ssm`). If they
install, it asks you to restart the session once, then uses the 370M backbone. If they do not
install — common on a T4 — it falls back to the 130M backbone on transformers' pure-PyTorch scan,
which is roughly ten times slower per parameter. 130M is still about 4,000x the paper's primary
model. It measures one forward pass, prints the expected runtime, and shortens phase A to fit
`TIME_BUDGET_H` (3 hours).

## Reading the output

- **Screen:** frozen-backbone accuracy per condition, forced choice among each prompt's own
  candidates, so emitting a plausible token cannot pass as recall. Viable at ≥ `VIABLE_ACC`.
- **Phase A:** accuracy, plus `D_attn` and `D_ssm` by resampling, plus accuracy with attention
  deleted. A futility check stops early if attention is not learning.

Send back `STAGE1_SUMMARY.txt`.


In [ ]:
import hashlib, json, math, os, pathlib, random, subprocess, sys, time
from contextlib import contextmanager, nullcontext

PRESET = "full"          # "smoke" = 130M backbone, tiny budgets
USE_DRIVE, DRIVE_DIR = True, "/content/drive/MyDrive/mamba_attn_route"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = DRIVE_DIR
else:
    WORKDIR = os.environ.get("MAMBA_WORKDIR", str(pathlib.Path.cwd() / "mamba_attn_route"))

from importlib.metadata import version, PackageNotFoundError


def _ver(pkg):
    try:
        return tuple(int("".join(c for c in p if c.isdigit()) or 0) for p in version(pkg).split(".")[:2])
    except PackageNotFoundError:
        return (0, 0)


if _ver("transformers") < (4, 43):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "accelerate"], check=True)
    raise SystemExit("transformers upgraded. Runtime -> Restart session, then Run all again.")
if _ver("datasets") == (0, 0):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)

# Optional fast Mamba kernels. Without them transformers uses a pure-PyTorch scan that is ~10x
# slower and much heavier. The install compiles CUDA code and often fails; that is fine, the
# notebook then sizes itself for the slow path. Set TRY_KERNELS=False to skip it entirely.
TRY_KERNELS = True
import importlib.util as _ilu
HAVE_KERNELS = all(_ilu.find_spec(m) is not None for m in ("mamba_ssm", "causal_conv1d"))
if TRY_KERNELS and not HAVE_KERNELS and not os.environ.get("KERNEL_INSTALL_TRIED"):
    os.environ["KERNEL_INSTALL_TRIED"] = "1"
    print("installing causal-conv1d and mamba-ssm (several minutes; failure is harmless)...", flush=True)
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "causal-conv1d", "mamba-ssm"],
                       capture_output=True, text=True)
    if r.returncode == 0 and all(_ilu.find_spec(m) is not None for m in ("mamba_ssm", "causal_conv1d")):
        raise SystemExit("fast kernels installed. Runtime -> Restart session, then Run all again.")
    print("fast kernels unavailable; continuing on the slow PyTorch path", flush=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib as mpl
import matplotlib.pyplot as plt
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM


def P(full, smoke):
    return full if PRESET == "full" else smoke


# Tried in order; the first that loads is used. With the fast kernels the 370M model is
# affordable; without them we start at 130M, which is still ~4,000x the paper's primary model.
CANDIDATES = (P(["AntonV/mamba2-370m-hf", "state-spaces/mamba-370m-hf", "AntonV/mamba2-130m-hf",
                 "state-spaces/mamba-130m-hf"], ["AntonV/mamba2-130m-hf", "state-spaces/mamba-130m-hf"])
              if HAVE_KERNELS else
              ["AntonV/mamba2-130m-hf", "state-spaces/mamba-130m-hf", "AntonV/mamba2-370m-hf"])
DISTANCES   = P([16, 64, 256], [16, 64])      # filler tokens between the fact and the question
SCREEN_LOADS = P([1, 2, 4, 8], [1, 2])        # key-value pairs (1 = single passkey)
SCREEN_DIST = 64                              # distance used for the load sweep
N_SCREEN    = P(64, 16)
N_EVAL      = P(32, 8)
N_ATTN_LAYERS = 8                             # evenly spaced layers that receive an attention branch
ATTN_HEADS  = 8
PHASE_A_STEPS = P(300, 20)
TRAIN_BATCH = P(8, 4)
EVAL_BATCH  = 8 if HAVE_KERNELS else 4
TIME_BUDGET_H = P(3.0, 0.2)          # phase A is shortened automatically to fit
LR, WARMUP, GRAD_CLIP = 3e-4, 30, 1.0
LM_W, LM_BATCH, LM_LEN = 0.5, 2, 256          # text replay, so the new route cannot wreck the LM
EVAL_EVERY = P(75, 10)
FUTILITY_STEP, FUTILITY_GAIN = P(150, 10), 0.05
TRAIN_LOADS = P([1, 2, 4, 8], [1, 2])         # phase A trains on a mixture, including hard loads
VIABLE_ACC, SOLVED_MARGIN = 0.80, 0.10
CKPT_MINUTES = P(20.0, 2.0)
SEED, EVAL_SEED, FORMAT_VERSION = 0, 999, 1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IS_CUDA = DEVICE.type == "cuda"
GPU_NAME = torch.cuda.get_device_name(0) if IS_CUDA else "cpu"
USE_BF16 = IS_CUDA and torch.cuda.get_device_capability()[0] >= 8


def amp():
    return torch.autocast(device_type="cuda", dtype=torch.bfloat16) if USE_BF16 else nullcontext()


def sync():
    if IS_CUDA:
        torch.cuda.synchronize()


WORK = pathlib.Path(WORKDIR).expanduser()
print(f"{GPU_NAME} | {'bf16 autocast' if USE_BF16 else 'float32'} | torch {torch.__version__} | "
      f"transformers {transformers.__version__} | fast Mamba kernels: {HAVE_KERNELS}")


In [ ]:
tok = model = MODEL = None
for cand in CANDIDATES:
    try:
        t0 = time.time()
        tok = AutoTokenizer.from_pretrained(cand)
        model = AutoModelForCausalLM.from_pretrained(cand, torch_dtype=torch.float32)
        MODEL = cand
        print(f"loaded {cand} in {time.time() - t0:.0f} s")
        break
    except Exception as e:
        print(f"could not load {cand}: {type(e).__name__}: {str(e)[:160]}")
assert model is not None, "no candidate backbone could be loaded"
model = model.to(DEVICE).eval()
model.config.use_cache = False
for p in model.parameters():
    p.requires_grad_(False)

TAG = f"{MODEL.split('/')[-1]}_{PRESET}"
RESDIR, CKPTDIR, FIGDIR, TABDIR = (WORK / f"{n}_{TAG}" for n in ("results", "checkpoints", "figures", "tables"))
for _d in (RESDIR, CKPTDIR, FIGDIR, TABDIR):
    _d.mkdir(parents=True, exist_ok=True)


def keyof(spec):
    return hashlib.md5(json.dumps(spec, sort_keys=True, default=str).encode()).hexdigest()[:16]


class Records:
    def __init__(self, path):
        self.path, self.recs = path, {}
        if path.exists():
            for line in path.read_text().splitlines():
                try:
                    r = json.loads(line); self.recs[r["_key"]] = r
                except (json.JSONDecodeError, KeyError):
                    pass

    def get(self, spec):
        return self.recs.get(keyof(spec))

    def put(self, spec, payload):
        r = {"_key": keyof(spec), "_spec": spec, "_t": time.time(), **payload}
        self.recs[r["_key"]] = r
        with open(self.path, "a") as f:
            f.write(json.dumps(r, default=float) + "\n"); f.flush(); os.fsync(f.fileno())
        return r


def ckpt_save(name, obj):
    p = CKPTDIR / f"{name}.pt"
    torch.save(obj, p.with_suffix(".tmp")); os.replace(p.with_suffix(".tmp"), p)


def ckpt_load(name):
    p = CKPTDIR / f"{name}.pt"
    if p.exists():
        try:
            return torch.load(p, map_location="cpu", weights_only=False)
        except Exception as e:
            print(f"  checkpoint unreadable ({type(e).__name__})")
    return None


def write_once(path, obj):
    if not path.exists():
        path.write_text(json.dumps(obj, indent=1))
    return json.loads(path.read_text())


REC = Records(RESDIR / "records.jsonl")
BASE_SPEC = {"model": MODEL, "seed": SEED, "format": FORMAT_VERSION}


def find_layers(m):
    n = m.config.num_hidden_layers
    hits = [(name, mod) for name, mod in m.named_modules() if isinstance(mod, nn.ModuleList) and len(mod) == n]
    assert hits, f"no ModuleList of length {n}"
    return hits[0]


LAYER_PATH, LAYERS = find_layers(model)
N_LAYERS = len(LAYERS)
D_MODEL = model.config.hidden_size
assert D_MODEL % ATTN_HEADS == 0
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters | {N_LAYERS} layers at "
      f"'{LAYER_PATH}' | hidden {D_MODEL} | block children "
      f"{[(n, type(c).__name__) for n, c in LAYERS[0].named_children()]}")
print(f"work dir {WORK} | results {RESDIR.name}")


In [ ]:
NOUNS = """apple river table window garden doctor pencil island forest bottle mirror candle bridge castle rocket
tiger lemon piano jacket helmet market pocket village teacher engine planet carpet button ticket basket hammer
ladder wallet bucket camera coffee cookie dragon farmer finger flower guitar hunter kitten letter monkey needle
orange parrot pillow puzzle rabbit saddle sister spider statue sugar thunder tomato turtle wagon wizard anchor
arrow badge banner barrel beach beard bell belt bench blanket boat bone book boot bread brick brush cabin cable
cake canal cannon card cart chain chair cheese cherry chest chicken church cloud coat coin comet cotton crown cup
curtain desk diamond dinner dog door drum eagle egg elbow fence field flag flute fork fox frog glass glove goat
grape hat honey horse house iron jar jewel kettle key kite knife lake lamp leaf lion lock map mask milk moon
mouse nail nest net oven owl paint palace paper pearl pepper pig pipe plate pond pot queen rain ring road robot
roof rope rose salt scarf ship shirt shoe shovel sink skirt snake soap sock spoon star stone storm sword tent
tower train tree truck tunnel violin wall whale wheel wolf yard""".split()
ADJS = """red blue green yellow purple brown black white pink gray golden silent happy angry brave calm clever
cold dark eager early empty fair fancy fast fierce gentle giant grand great heavy hollow humble icy jolly kind
large late lazy little lonely loud lucky mild modern narrow neat noble odd old plain polite proud quick quiet
rapid rare rich rough round royal rusty sacred safe sharp shiny short shy simple sleepy slow small smooth soft
solid sour spicy steep sticky stormy strange strict strong sunny sweet swift tall tame thick thin tidy tiny tough
ugly vast warm weak wet wide wild wise young bitter bold bright broad busy cheap crisp cruel cute damp deep dry
dusty faint false fresh frozen full funny grumpy hairy harsh hot huge juicy keen lean lively loyal mad merry
messy misty muddy nasty naughty nervous nice pale pure rigid ripe rude salty scary shallow silly skinny spotty
stale stiff sturdy tender tense tricky vivid weird wooden zany""".split()


def single_token(words, exclude=()):
    ids = []
    for w in words:
        e = tok.encode(" " + w, add_special_tokens=False)
        if len(e) == 1 and e[0] not in ids and e[0] not in exclude:
            ids.append(e[0])
    return ids


KEY_IDS = single_token(NOUNS)
VAL_IDS = single_token(ADJS, exclude=set(KEY_IDS))
N_CAND = 16
assert len(KEY_IDS) >= max(SCREEN_LOADS) and len(VAL_IDS) >= N_CAND, (len(KEY_IDS), len(VAL_IDS))
print(f"single-token pools: {len(KEY_IDS)} keys, {len(VAL_IDS)} values | {N_CAND} candidates per prompt "
      f"(chance {1 / N_CAND:.3f})")


def piece(s):
    return tok.encode(s, add_special_tokens=False)


BOS = [tok.bos_token_id] if tok.bos_token_id is not None else []
_WIKI = None


def wiki_tokens():
    global _WIKI
    if _WIKI is None:
        from datasets import load_dataset
        try:
            ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="validation")
        except Exception:
            ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="validation")
        _WIKI = piece("\n\n".join(t for t in ds["text"] if t.strip()))
    return _WIKI


def make_prompts(load, dist, n, seed):
    # natural-text filler between the facts and the question; one length per condition (no padding)
    W = wiki_tokens()
    rng = np.random.default_rng([seed, load, dist, FORMAT_VERSION])
    X, Y, C = [], [], []
    for _ in range(n):
        ks = rng.choice(len(KEY_IDS), load, replace=False)
        cand = rng.choice(len(VAL_IDS), N_CAND, replace=False)
        vs = cand[:load]
        q = int(rng.integers(load))
        ids = BOS + piece("Note these facts.\n")
        for k, v in zip(ks, vs):
            ids += piece("The") + [KEY_IDS[k]] + piece(" is") + [VAL_IDS[v]] + piece(".\n")
        s = int(rng.integers(len(W) - dist - 1))
        ids += piece("\n") + W[s:s + dist] + piece("\nQuestion: the") + [KEY_IDS[ks[q]]] + piece(" is")
        X.append(ids); Y.append(VAL_IDS[vs[q]]); C.append([VAL_IDS[c] for c in cand])
    L = max(len(x) for x in X)
    keep = [i for i, x in enumerate(X) if len(x) == L]
    return (torch.tensor([X[i] for i in keep]), torch.tensor([Y[i] for i in keep]),
            torch.tensor([C[i] for i in keep]))


_x, _y, _c = make_prompts(2, 16, 2, SEED)
_t = tok.decode(_x[0])
print("\nexample (2 pairs, 16 filler tokens):\n" + (_t if len(_t) < 600 else _t[:300] + " [...] " + _t[-300:])
      + f"\n   -> answer{tok.decode([int(_y[0])])}")


In [ ]:
ACTIVE = {}          # route interventions; populated by the intervene() context manager (next cell)


def slices(n, B):
    B = max(2, B)
    sl = [(s, min(s + B, n)) for s in range(0, n, B)]
    if len(sl) > 1 and sl[-1][1] - sl[-1][0] == 1:
        sl[-2] = (sl[-2][0], n); sl.pop()
    return sl


def with_oom_retry(fn, B, label):
    while True:
        try:
            return fn(B)
        except torch.cuda.OutOfMemoryError:
            ACTIVE.clear()
            if IS_CUDA:
                torch.cuda.empty_cache()
            if B <= 2:
                raise
            B //= 2
            print(f"    out of memory in {label}; retrying with batch {B}", flush=True)


@torch.no_grad()
def last_logits(x):
    with amp():
        return model(input_ids=x, use_cache=False).logits[:, -1, :].float()


def forced_choice(lg, cand):
    return cand.gather(1, torch.gather(lg, 1, cand).argmax(-1, keepdim=True)).squeeze(1)


@torch.no_grad()
def screen_condition(load, dist, B):
    X, Y, C = make_prompts(load, dist, N_SCREEN, SEED)
    hit = top = incand = n = 0
    for i, j in slices(len(X), B):
        x, y, c = X[i:j].to(DEVICE), Y[i:j].to(DEVICE), C[i:j].to(DEVICE)
        lg = last_logits(x)
        hit += int((forced_choice(lg, c) == y).sum())
        t = lg.argmax(-1)
        top += int((t == y).sum()); incand += int((t[:, None] == c).any(1).sum()); n += len(x)
    return {"load": load, "dist": dist, "tokens": int(X.shape[1]), "n": n, "chance": 1.0 / N_CAND,
            "acc": hit / n, "top1": top / n, "in_cand": incand / n}


sync(); _t0 = time.time()
_probe = make_prompts(1, SCREEN_DIST, EVAL_BATCH, SEED)[0].to(DEVICE)
with torch.no_grad():
    last_logits(_probe)
sync()
SEC_PER_FWD = time.time() - _t0
del _probe
if IS_CUDA:
    torch.cuda.empty_cache()
print(f"forward pass: {SEC_PER_FWD:.2f} s at batch {EVAL_BATCH} "
      f"({EVAL_BATCH / SEC_PER_FWD:.1f} prompts/s){'' if HAVE_KERNELS else '  [slow PyTorch scan]'}")

SCREEN_CONDS = ([(1, d) for d in DISTANCES] + [(l, SCREEN_DIST) for l in SCREEN_LOADS if l > 1])
_screen_min = len(SCREEN_CONDS) * N_SCREEN / EVAL_BATCH * SEC_PER_FWD / 60
print(f"screen should take about {_screen_min:.0f} min")
SCREEN = []
print(f"\nfrozen backbone screen ({N_SCREEN} prompts per condition, chance {1 / N_CAND:.3f}):")
print(f"{'pairs':>6}{'filler':>8}{'tok':>6}{'acc':>8}{'top1':>7}{'inCand':>8}")
for load, dist in SCREEN_CONDS:
    spec = {**BASE_SPEC, "block": "screen", "load": load, "dist": dist, "n": N_SCREEN, "cand": N_CAND}
    r = REC.get(spec) or REC.put(spec, with_oom_retry(lambda B, l=load, d=dist: screen_condition(l, d, B),
                                                      EVAL_BATCH * 2, f"screen {load}/{dist}"))
    SCREEN.append(r)
    print(f"{load:>6}{dist:>8}{r['tokens']:>6}{r['acc']:>8.3f}{r['top1']:>7.3f}{r['in_cand']:>8.3f}", flush=True)

SC = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")} for r in SCREEN])
SC["viable"] = SC.acc >= VIABLE_ACC
viable = SC[SC.viable]
nonviable = SC[~SC.viable]
BOUNDARY = write_once(RESDIR / "boundary.json", {
    "created": time.strftime("%Y-%m-%d %H:%M:%S %Z"), "model": MODEL, "rule": f"viable if accuracy >= {VIABLE_ACC}",
    "conditions": [{"load": int(r.load), "dist": int(r.dist), "acc": float(r.acc), "viable": bool(r.viable)}
                   for r in SC.itertuples()],
    "viable": [[int(r.load), int(r.dist)] for r in viable.itertuples()],
    "nonviable": [[int(r.load), int(r.dist)] for r in nonviable.itertuples()]})
print(f"\nviable (recurrent route alone solves it): {BOUNDARY['viable'] or 'none'}")
print(f"not viable: {BOUNDARY['nonviable'] or 'none'}   [boundary recorded {BOUNDARY['created']}]")
SCREEN_OK = len(BOUNDARY["viable"]) > 0
if not SCREEN_OK:
    print("\nThe frozen backbone does not solve any tested condition, so there is no viable recurrent "
          "route to steer toward. Stage 1 stops here; send the screen back.")


In [ ]:
# Wrap every block so its two contributions are separable:  h <- h + ssm_delta(h) + attn(h)
# attn is present only at the selected layers and its output projection starts at zero, so the
# wrapped model is numerically identical to the pretrained one until training begins.
def apply_iv(route, idx, t):
    spec = ACTIVE.get(route)
    if spec is None or t is None:
        return t
    kind = spec["kind"]
    if kind == "record":
        spec["store"].setdefault(idx, []).append(t.detach().clone())   # native precision: identity
        # resampling must reproduce the clean pass exactly, and a float16 cast breaks that over 48 layers
        return t
    if kind == "zero":
        return torch.zeros_like(t)
    if kind == "cache":
        return spec["store"][idx].to(t.dtype)[spec["perm"]]
    if kind == "mask":
        return t * spec["keep"].to(t.dtype)[:, None, None]
    raise ValueError(kind)


class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.w, self.eps = nn.Parameter(torch.ones(d)), eps

    def forward(self, x):
        return self.w * x * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).to(x.dtype)


def rope(T, dh, device, base=10000.0):
    inv = 1.0 / (base ** (torch.arange(0, dh, 2, device=device).float() / dh))
    f = torch.outer(torch.arange(T, device=device).float(), inv)
    return f.cos(), f.sin()


def apply_rope(x, cos, sin):
    x1, x2 = x[..., ::2], x[..., 1::2]
    cos, sin = cos[None, None].to(x.dtype), sin[None, None].to(x.dtype)
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], -1).flatten(-2)


class AttentionRoute(nn.Module):
    def __init__(self, d, n_heads):
        super().__init__()
        self.h, self.dh = n_heads, d // n_heads
        self.norm = RMSNorm(d)
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.o = nn.Linear(d, d, bias=False)
        nn.init.normal_(self.qkv.weight, std=0.02)
        nn.init.zeros_(self.o.weight)              # the added route starts silent

    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(self.norm(x)).view(B, T, 3, self.h, self.dh).permute(2, 0, 3, 1, 4)
        cos, sin = rope(T, self.dh, x.device)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o(y.transpose(1, 2).reshape(B, T, D))


class DualRouteBlock(nn.Module):
    def __init__(self, base, idx, attn=None):
        super().__init__()
        self.base, self.idx, self.attn = base, idx, attn

    def forward(self, hidden_states, *args, **kwargs):
        out = self.base(hidden_states, *args, **kwargs)
        h = out[0] if isinstance(out, tuple) else out
        ssm = apply_iv("ssm", self.idx, h - hidden_states)
        a = apply_iv("attn", self.idx, self.attn(hidden_states) if self.attn is not None else None)
        new = hidden_states + ssm + (a if a is not None else 0.0)
        return (new,) + tuple(out[1:]) if isinstance(out, tuple) else new


ATTN_LAYERS = sorted(set(np.linspace(0, N_LAYERS - 1, N_ATTN_LAYERS).round().astype(int).tolist()))
if not isinstance(LAYERS[0], DualRouteBlock):
    for i in range(N_LAYERS):
        route = AttentionRoute(D_MODEL, ATTN_HEADS).to(DEVICE) if i in ATTN_LAYERS else None
        LAYERS[i] = DualRouteBlock(LAYERS[i], i, route)
ATTN_PARAMS = [p for blk in LAYERS if blk.attn is not None for p in blk.attn.parameters()]
for p in ATTN_PARAMS:
    p.requires_grad_(True)
N_ATTN_PARAMS = sum(p.numel() for p in ATTN_PARAMS)


@contextmanager
def intervene(**routes):
    ACTIVE.clear()
    ACTIVE.update({k: v for k, v in routes.items() if v is not None})
    try:
        yield
    finally:
        ACTIVE.clear()


def record_store():
    return {"attn": {}, "ssm": {}}


def stack_store(store):
    return {r: {i: torch.cat(v) if len(v) > 1 else v[0] for i, v in d.items()} for r, d in store.items()}


print(f"attention route added at layers {ATTN_LAYERS} ({N_ATTN_PARAMS / 1e6:.1f}M new parameters, "
      f"output projection zero-initialised)")


In [ ]:
# Self-test: the wrapped model is unchanged at initialisation, and both routes are interceptable.
model.eval()
_x = make_prompts(2, 16, 4, SEED)[0].to(DEVICE)
with torch.no_grad(), amp():
    _base = model(input_ids=_x, use_cache=False).logits[:, -1, :].float()
_tol = (1e-4 if not USE_BF16 else 5e-3) * max(1.0, float(_base.abs().max()))
_st = record_store()
with intervene(attn={"kind": "record", "store": _st["attn"]}, ssm={"kind": "record", "store": _st["ssm"]}):
    _clean = last_logits(_x)
assert float((_base - _clean).abs().max()) <= _tol, "recording changed the output"
assert len(_st["ssm"]) == N_LAYERS and len(_st["attn"]) == len(ATTN_LAYERS), \
    f"intercepted {len(_st['ssm'])} ssm and {len(_st['attn'])} attention layers"
_st = stack_store(_st)
_id = torch.arange(len(_x), device=DEVICE)
with intervene(ssm={"kind": "cache", "store": _st["ssm"], "perm": _id}):
    assert float((last_logits(_x) - _clean).abs().max()) <= _tol, "identity resample changed the output"
with intervene(attn={"kind": "zero"}):
    assert float((last_logits(_x) - _clean).abs().max()) <= _tol, "zeroing the silent route changed the output"
with intervene(ssm={"kind": "cache", "store": _st["ssm"], "perm": torch.roll(_id, 1)}):
    assert float((last_logits(_x) - _clean).abs().max()) > 10 * _tol, "resampling the SSM route had no effect"
_out_proj = next(blk.attn.o.weight for blk in LAYERS if blk.attn is not None)
_saved = _out_proj.detach().clone()
with torch.no_grad():
    _out_proj.add_(torch.randn_like(_out_proj) * 0.01)
assert float((last_logits(_x) - _clean).abs().max()) > 10 * _tol, "the attention route cannot affect the output"
with torch.no_grad():
    _out_proj.copy_(_saved)
assert float((last_logits(_x) - _clean).abs().max()) <= _tol, "restoring the attention route failed"
del _st, _saved
if IS_CUDA:
    torch.cuda.empty_cache()
print(f"self-test passed: wrapped model identical at init (tol {_tol:.1e}), {N_LAYERS} SSM and "
      f"{len(ATTN_LAYERS)} attention interception points, resampling bites")


In [ ]:
# Phase A: train ONLY the new attention route on a mixture of loads, backbone frozen.
# Question: does the added route take over conditions the backbone can already solve alone?
@torch.no_grad()
def evaluate(B=EVAL_BATCH):
    model.eval()
    out = {}
    for load, dist in EVAL_CONDS:
        X, Y, C = make_prompts(load, dist, N_EVAL, EVAL_SEED)
        tot = dict.fromkeys(["clean", "attn_res", "ssm_res", "no_attn", "donor"], 0.0)
        n = 0
        for i, j in slices(len(X), B):
            x, y, c = X[i:j].to(DEVICE), Y[i:j].to(DEVICE), C[i:j].to(DEVICE)
            perm = torch.roll(torch.arange(len(x), device=DEVICE), 1)
            st = record_store()
            with intervene(attn={"kind": "record", "store": st["attn"]},
                           ssm={"kind": "record", "store": st["ssm"]}):
                lg = last_logits(x)
            st = stack_store(st)
            tot["clean"] += int((forced_choice(lg, c) == y).sum())
            with intervene(attn={"kind": "cache", "store": st["attn"], "perm": perm}):
                p_a = forced_choice(last_logits(x), c)
            tot["attn_res"] += int((p_a == y).sum()); tot["donor"] += int((p_a == y[perm]).sum())
            with intervene(ssm={"kind": "cache", "store": st["ssm"], "perm": perm}):
                tot["ssm_res"] += int((forced_choice(last_logits(x), c) == y).sum())
            with intervene(attn={"kind": "zero"}):
                tot["no_attn"] += int((forced_choice(last_logits(x), c) == y).sum())
            del st
            n += len(x)
            if IS_CUDA:
                torch.cuda.empty_cache()
        m = {k: v / n for k, v in tot.items()}
        head = max(m["clean"] - 1.0 / N_CAND, 1e-9)
        tag = f"L{load}_d{dist}"
        out[f"acc_{tag}"] = m["clean"]
        out[f"no_attn_{tag}"] = m["no_attn"]
        out[f"donor_{tag}"] = m["donor"]
        for r, key in (("attn", "attn_res"), ("ssm", "ssm_res")):
            out[f"D_{r}_{tag}"] = (float("nan") if m["clean"] - 1.0 / N_CAND < 0.10
                                   else float(np.clip((m["clean"] - m[key]) / head, 0.0, 1.0)))
    model.train()
    return out


def line(ev):
    return "  ".join(f"{t}: acc {ev[f'acc_{t}']:.2f} noAttn {ev[f'no_attn_{t}']:.2f} "
                     f"Da {ev[f'D_attn_{t}']:.2f} Ds {ev[f'D_ssm_{t}']:.2f}" for t in EVAL_TAGS)


_wiki_train = None


def replay(rng):
    global _wiki_train
    if _wiki_train is None:
        W = wiki_tokens()
        n = len(W) // LM_LEN
        _wiki_train = torch.tensor(W[: n * LM_LEN]).view(n, LM_LEN)
    x = _wiki_train[rng.integers(0, len(_wiki_train), LM_BATCH)].to(DEVICE)
    with amp():
        lg = model(input_ids=x, use_cache=False).logits[:, :-1]
    return F.cross_entropy(lg.float().reshape(-1, lg.shape[-1]), x[:, 1:].reshape(-1))


EVAL_CONDS = ([tuple(c) for c in BOUNDARY["viable"][:2]] + [tuple(c) for c in BOUNDARY["nonviable"][:1]]) \
    if SCREEN_OK else []
EVAL_TAGS = [f"L{l}_d{d}" for l, d in EVAL_CONDS]

# Size phase A to the time budget using the measured forward-pass cost. One evaluation is
# 4 interventions x N_EVAL prompts x len(EVAL_CONDS); one training step is ~3 forward-equivalents.
if SCREEN_OK:
    eval_min = 4 * len(EVAL_CONDS) * N_EVAL / EVAL_BATCH * SEC_PER_FWD / 60
    step_min = 3 * SEC_PER_FWD * TRAIN_BATCH / EVAL_BATCH / 60
    for cand in (PHASE_A_STEPS, 200, 150, 100, 60):
        n_evals = 2 + cand // max(EVAL_EVERY, 1)
        if cand * step_min + n_evals * eval_min <= TIME_BUDGET_H * 60:
            PHASE_A_STEPS = cand
            break
    else:
        PHASE_A_STEPS = 60
    EVAL_EVERY = max(25, PHASE_A_STEPS // 4)
    FUTILITY_STEP = max(25, PHASE_A_STEPS // 2)
    n_evals = 2 + PHASE_A_STEPS // EVAL_EVERY
    print(f"phase A: {PHASE_A_STEPS} steps, evaluation every {EVAL_EVERY} (futility check at "
          f"{FUTILITY_STEP}) -> about {(PHASE_A_STEPS * step_min + n_evals * eval_min) / 60:.1f} h "
          f"(budget {TIME_BUDGET_H} h)")

PHASE_A = None
if SCREEN_OK:
    spec = {**BASE_SPEC, "block": "phaseA", "steps": PHASE_A_STEPS, "lr": LR, "batch": TRAIN_BATCH,
            "train_loads": TRAIN_LOADS, "dist": SCREEN_DIST, "attn_layers": ATTN_LAYERS, "heads": ATTN_HEADS,
            "replay": [LM_W, LM_BATCH, LM_LEN], "eval_conds": EVAL_CONDS}
    KEY = keyof(spec)
    PHASE_A = REC.get(spec)
    if PHASE_A is None:
        opt = torch.optim.AdamW(ATTN_PARAMS, lr=LR, betas=(0.9, 0.95), weight_decay=0.0)
        start, trace, rt, rs = 0, [], 0.0, 0
        ck = ckpt_load(f"phaseA_{KEY}")
        if ck is not None:
            for blk, sd in zip([b for b in LAYERS if b.attn is not None], ck["attn"]):
                blk.attn.load_state_dict(sd)
            opt.load_state_dict(ck["opt"]); start, trace, rt, rs = ck["step"], ck["trace"], ck["rt"], ck["rs"]
            print(f"[phase A] resumed at step {start}/{PHASE_A_STEPS}")
        if start == 0:
            ev = with_oom_retry(lambda B: evaluate(B), EVAL_BATCH, "evaluation")
            trace.append({"step": 0, "loss": float("nan"), "lm": float("nan"), **ev})
            print(f"[phase A] step 0 (attention silent): {line(ev)}", flush=True)
        rng = np.random.default_rng([SEED, start])
        model.train()
        last_ck = time.time()
        losses, lms = [], []
        for step in range(start + 1, PHASE_A_STEPS + 1):
            sync(); t0 = time.time()
            for g in opt.param_groups:
                g["lr"] = LR * min(1.0, step / WARMUP)
            load = TRAIN_LOADS[step % len(TRAIN_LOADS)]
            X, Y, _ = make_prompts(load, SCREEN_DIST, TRAIN_BATCH, int(rng.integers(2 ** 31)))
            x, y = X.to(DEVICE), Y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with amp():
                lg = model(input_ids=x, use_cache=False).logits[:, -1, :]
            loss = F.cross_entropy(lg.float(), y)
            loss.backward()
            lm = replay(rng)
            (LM_W * lm).backward()
            nn.utils.clip_grad_norm_(ATTN_PARAMS, GRAD_CLIP)
            opt.step()
            sync(); rt += time.time() - t0; rs += 1
            losses.append(float(loss.detach())); lms.append(float(lm.detach()))
            if step == start + 3:
                print(f"[phase A] {rt / rs:.1f} s/step, about {rt / rs * (PHASE_A_STEPS - step) / 60:.0f} min left",
                      flush=True)
            if step % EVAL_EVERY == 0 or step == PHASE_A_STEPS or step == FUTILITY_STEP:
                ev = with_oom_retry(lambda B: evaluate(B), EVAL_BATCH, "evaluation")
                trace.append({"step": step, "loss": float(np.mean(losses)), "lm": float(np.mean(lms)), **ev})
                print(f"[phase A] step {step} (task loss {np.mean(losses):.3f}, text {np.mean(lms):.3f}): "
                      f"{line(ev)}", flush=True)
                losses, lms = [], []
                model.train()
                if step == FUTILITY_STEP:
                    gain = max(ev[f"D_attn_{t}"] for t in EVAL_TAGS)
                    if not (gain > FUTILITY_GAIN):
                        print(f"[phase A] futility check: attention dependence still {gain:.3f} at step {step}; "
                              f"the added route is not taking over. Stopping.", flush=True)
                        break
            if step < PHASE_A_STEPS and time.time() - last_ck > 60 * CKPT_MINUTES:
                ckpt_save(f"phaseA_{KEY}", {"attn": [b.attn.state_dict() for b in LAYERS if b.attn is not None],
                                            "opt": opt.state_dict(), "step": step, "trace": trace, "rt": rt, "rs": rs})
                last_ck = time.time()
        ckpt_save(f"phaseA_final_{KEY}", {"attn": [b.attn.state_dict() for b in LAYERS if b.attn is not None],
                                          "trace": trace})
        PHASE_A = REC.put(spec, {"trace": trace, "sec_per_step": rt / max(rs, 1), "steps_done": len(trace)})
        (CKPTDIR / f"phaseA_{KEY}.pt").unlink(missing_ok=True)
    else:
        print("[phase A] finished earlier")


In [ ]:
lines = [f"=== STAGE 1: PRETRAINED RECURRENT BACKBONE + ADDED ATTENTION ROUTE | {time.strftime('%Y-%m-%d %H:%M')} ===",
         f"model {MODEL} | {GPU_NAME} | {'bf16 autocast' if USE_BF16 else 'float32'} | {N_LAYERS} layers, "
         f"hidden {D_MODEL} | attention at layers {ATTN_LAYERS} ({N_ATTN_PARAMS / 1e6:.1f}M new parameters)",
         f"{N_SCREEN} prompts per screened condition, {N_EVAL} per evaluated condition, {N_CAND} candidates "
         f"(chance {1 / N_CAND:.3f})", "",
         "-- frozen backbone screen (can recurrence alone do it?) --",
         SC[["load", "dist", "tokens", "acc", "top1", "in_cand", "viable"]].round(3).to_string(index=False),
         f"boundary recorded {BOUNDARY['created']}: viable {BOUNDARY['viable']}, not viable {BOUNDARY['nonviable']}"]

if SCREEN_OK and PHASE_A is not None:
    TR = pd.DataFrame(PHASE_A["trace"])
    TR.to_csv(TABDIR / "phaseA_trace.csv", index=False)
    cols = ["step", "loss", "lm"] + [f"{m}_{t}" for t in EVAL_TAGS
                                     for m in ("acc", "no_attn", "donor", "D_attn", "D_ssm")]
    first, last = TR.iloc[0], TR.iloc[-1]
    took_over = {t: (last[f"D_attn_{t}"] > 0.5 and last[f"D_attn_{t}"] > last[f"D_ssm_{t}"]) for t in EVAL_TAGS}
    solved = {t: last[f"acc_{t}"] >= VIABLE_ACC for t in EVAL_TAGS}
    viable_tags = [f"L{l}_d{d}" for l, d in BOUNDARY["viable"][:2]]
    ready = [t for t in viable_tags if t in took_over and took_over[t] and solved[t]]
    VERDICT = ("READY FOR STAGE 2" if ready else
               "ATTENTION DID NOT TAKE OVER" if not any(took_over.values()) else
               "PARTIAL: attention dominant only where the backbone is not viable")
    lines += ["", f"-- phase A: {PHASE_A['steps_done'] - 1} evaluations, {PHASE_A['sec_per_step']:.1f} s/step --",
              TR[cols].round(3).to_string(index=False), "",
              "per condition, start -> end:"]
    for t in EVAL_TAGS:
        lines.append(f"  {t:<10} acc {first[f'acc_{t}']:.3f} -> {last[f'acc_{t}']:.3f} | "
                     f"D_attn {first[f'D_attn_{t}']:.3f} -> {last[f'D_attn_{t}']:.3f} | "
                     f"D_ssm {first[f'D_ssm_{t}']:.3f} -> {last[f'D_ssm_{t}']:.3f} | "
                     f"acc without attention {first[f'no_attn_{t}']:.3f} -> {last[f'no_attn_{t}']:.3f} | "
                     f"donor {last[f'donor_{t}']:.3f}")
    lines += ["", f"conditions where attention took over and the task is solved: {ready or 'none'}",
              f"VERDICT: {VERDICT}"]

    mpl.rcParams.update({"figure.dpi": 130, "savefig.dpi": 300, "savefig.bbox": "tight", "font.size": 8,
                         "axes.spines.top": False, "axes.spines.right": False, "legend.fontsize": 7})
    fig, axes = plt.subplots(1, 3, figsize=(7.4, 2.4))
    for t in EVAL_TAGS:
        axes[0].plot(TR.step, TR[f"acc_{t}"], "-o", ms=3, label=t)
        axes[1].plot(TR.step, TR[f"D_attn_{t}"], "-o", ms=3, label=f"{t} attention")
        axes[1].plot(TR.step, TR[f"D_ssm_{t}"], ":s", ms=3, label=f"{t} SSM")
        axes[2].plot(TR.step, TR[f"no_attn_{t}"], "-o", ms=3, label=t)
    for ax, ttl in zip(axes, ("recall accuracy", "causal dependence", "accuracy without attention")):
        ax.set_xlabel("phase A step"); ax.set_title(ttl); ax.set_ylim(-0.03, 1.03)
    axes[0].axhline(1 / N_CAND, color="0.7", lw=0.6, ls=":")
    h, l = axes[1].get_legend_handles_labels()
    fig.tight_layout()
    fig.legend(h, l, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.0))
    for ext in ("pdf", "png"):
        fig.savefig(FIGDIR / f"stage1_phaseA.{ext}")
    plt.show()
else:
    lines += ["", "phase A not run (no viable condition in the screen)"]

SUMMARY = "\n".join(lines)
(TABDIR / "STAGE1_SUMMARY.txt").write_text(SUMMARY)
SC.to_csv(TABDIR / "screen.csv", index=False)
print(SUMMARY)
print(f"\nsaved {TABDIR / 'STAGE1_SUMMARY.txt'}")
